# SAC arrival_v2 — single U=1.5 cross + upstream control rerun (seed=42, 1M, s1)

**前情**：commit `f179c5b` / commit-after `1adee4e` 已闭环 arrival_v2 reward 在四个 benchmark 上的 vanilla SAC 验证（详见 [`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md)）：
- `single_u10_cross_tgt15` (s0/10-D, U=1.0, seed=46, 1M)         : 5/5 PASS
- `single_u15_upstream_tgt15` (s1/12-D, U=1.5, seed=46, **1.5M**): 5/5 PASS  ← P1 v6
- `tandem_u15_upstream_tgt15` (s1/12-D, U=1.5, seed=42, 1M)      : 5/5 PASS
- `sbs_u15_upstream_tgt15`    (s1/12-D, U=1.5, seed=42, 1M)      : 5/5 PASS

**本 notebook 任务（控制实验）**：作为 tandem/sbs 的拓扑横向对照，§3 P1 v6 (seed=46/1.5M) 与 §2 cross_u10 (U=1.0/seed=46) 在 seed/step/U 上都和 tandem/sbs 不一致，**confounded**。补两个**严格控制对照**：
- Phase 1 — `single_u15_cross_tgt15`:    单柱 cross_stream, U=1.5, seed=42, **1.0M**
- Phase 2 — `single_u15_upstream_tgt15`: 单柱 upstream,     U=1.5, seed=42, **1.0M**

补完后，§7.3 横向对照表的所有四组运行严格控制（s1 / k4 / U=1.5 / target=1.5 / arrival_v2 / seed=42 / 1M），唯一变量 = topology × geometry：

| Run | Geometry | Topology |
|---|---|---|
| Phase 1 (本 notebook) | cross_stream | single |
| Phase 2 (本 notebook) | upstream     | single |
| §7.1 tandem  | upstream     | double tandem (G/D=3.5) |
| §7.2 sbs     | upstream     | double side-by-side (G/D=3.5) |

**配置矩阵**：

| Phase | benchmark | flow | obs | total_steps | 预算 |
|---|---|---|---|---:|---:|
| 1 | `single_u15_cross_tgt15`    | `wake_v8_U1p50_Re250` | s1 / 12-D | 1.0M | ~2.5h L4 |
| 2 | `single_u15_upstream_tgt15` | `wake_v8_U1p50_Re250` | s1 / 12-D | 1.0M | ~2.5h L4 |

**Gate**（每个 phase 独立判定，与 P1 v6 §8.3 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Single seed**：`seed=42`（与 tandem/sbs 同步）。如果 1.0M 收敛了就闭环；如果某一 phase eval 曲线尚在上升，把对应的 `TOTAL_STEPS` 改大（如 1.5M）后重跑同一 cell — `--resume` 会自动从 `trainer_state.json` 续训。

**注意**：Phase 2 (`single_u15_upstream_tgt15`) 与 §3 P1 v6 同一 benchmark，但 seed=42（vs §3 seed=46）+ budget=1M（vs §3 1.5M）。两次 run 输出在不同 seed 子目录下，互不覆盖。

**输出根**：
- Phase 1: `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42/`
- Phase 2: `experiments/arrival_v2_prototype/single_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42/`

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout）。

## 0. GPU sanity

In [ ]:
!nvidia-smi | head -10
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

## 1. Mount Drive + cwd

和 commit `f179c5b` / `1adee4e` 同一个 working clone（含 arrival_v2 实现 + train_sac resume bug 修复 + tandem/sbs notebook 闭环）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR

## 2. 通用配置 — Phase 1 (single_cross) + Phase 2 (single_upstream)

唯一变量是 `task_geometry` (cross_stream vs upstream) + `benchmark_key`。所有 SAC / env / reward 超参与 tandem/sbs 完全一致（s1 / k4 / seed=42 / num_envs=6 / arrival_v2），便于做严格控制对照。

若某 phase 1.0M 不够，把对应 `TOTAL_STEPS` 改成 `1_500_000` 或 `2_000_000` 后重跑该 phase 的 train cell — skip/resume 逻辑会自动续训。

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== 全局 SAC / env config (两个 phase 共用) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's1'
HISTORY_LENGTH = 4
TARGET_SPEED = 1.5
SEED = 42

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Single U=1.5 flow file (shared by both phases)
SHARED_FLOW_PATH = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Phase 1 — single_cross ====
C_BENCHMARK_KEY = 'single_u15_cross_tgt15'
C_TASK_GEOMETRY = 'cross_stream'
C_FLOW_PATH = SHARED_FLOW_PATH
C_TOTAL_STEPS = 1_000_000           # 起步 1M;不够就改大重跑此 cell
C_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
C_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
C_MANIFEST_PATH = Path(f'benchmarks/{C_BENCHMARK_KEY}.json')
C_RUN_ROOT_STR = str(C_RUN_ROOT)
C_CKPT_ROOT_STR = str(C_CKPT_ROOT)
C_MANIFEST_PATH_STR = str(C_MANIFEST_PATH)

# ==== Phase 2 — single_upstream ====
U_BENCHMARK_KEY = 'single_u15_upstream_tgt15'
U_TASK_GEOMETRY = 'upstream'
U_FLOW_PATH = SHARED_FLOW_PATH
U_TOTAL_STEPS = 1_000_000           # 起步 1M
U_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
U_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
U_MANIFEST_PATH = Path(f'benchmarks/{U_BENCHMARK_KEY}.json')
U_RUN_ROOT_STR = str(U_RUN_ROOT)
U_CKPT_ROOT_STR = str(U_CKPT_ROOT)
U_MANIFEST_PATH_STR = str(U_MANIFEST_PATH)

os.environ['PYTHONUNBUFFERED'] = '1'

print('---- Phase 1 (single_cross) ----')
print(f'  benchmark    = {C_BENCHMARK_KEY}')
print(f'  flow         = {C_FLOW_PATH}')
print(f'  probe        = {PROBE_LAYOUT} / {C_TASK_GEOMETRY} / target={TARGET_SPEED}')
print(f'  total_steps  = {C_TOTAL_STEPS:,} (initial; bump and re-run if not converged)')
print(f'  run_root     = {C_RUN_ROOT}')
print(f'  ckpt_root    = {C_CKPT_ROOT}')
print()
print('---- Phase 2 (single_upstream) ----')
print(f'  benchmark    = {U_BENCHMARK_KEY}')
print(f'  flow         = {U_FLOW_PATH}')
print(f'  probe        = {PROBE_LAYOUT} / {U_TASK_GEOMETRY} / target={TARGET_SPEED}')
print(f'  total_steps  = {U_TOTAL_STEPS:,} (initial)')
print(f'  run_root     = {U_RUN_ROOT}')
print(f'  ckpt_root    = {U_CKPT_ROOT}')
print()
print(f'  seed={SEED}  num_envs={NUM_ENVS}  eval_every={EVAL_EVERY:,}  eval_episodes={EVAL_EPISODES}')
print()
print('NOTE: Phase 2 与 §3 P1 v6 同 benchmark; seed=42 子目录与 §3 seed=46 互不覆盖。')

## 3. Preflight — flow files / arrival_v2 candidate gate / reward unit tests / manifests

停止条件：
- 任一 phase flow 缺失 → raise（用户已确认 Drive 有；此处兜底）
- `validate_arrival_v2_candidate` 失败 → reward 公式不再满足 discounted unsafe-shortcut / terminal dominance
- `test_reward_objective.py` 失败 → reward 实现退化
- 任一 manifest 生成失败 → eval 不可重复

In [ ]:
# 两个 phase 共用同一个 single U=1.5 flow file; 检查一次即可
p = Path(SHARED_FLOW_PATH)
if not p.exists():
    raise FileNotFoundError(f'missing single U=1.5 flow file: {p}')
print(f'[OK] shared flow: {p} ({p.stat().st_size / 1e6:.1f} MB)')

In [ ]:
!python -u -m scripts.validate_arrival_v2_candidate

In [ ]:
!python -u -m pytest tests/test_reward_objective.py -q

In [ ]:
for key, mp in ((C_BENCHMARK_KEY, C_MANIFEST_PATH), (U_BENCHMARK_KEY, U_MANIFEST_PATH)):
    if not mp.exists():
        !python -u -m scripts.generate_standard_benchmarks --benchmarks {key} --episodes {EVAL_EPISODES}
    if not mp.exists():
        raise FileNotFoundError(f'manifest not generated: {mp}')
    print(f'[OK] manifest ready: {mp}')

## 4. Phase 1 — single_cross train (1.0M, fresh start with skip/resume)

Skip/resume 逻辑：
- `env_step >= C_TOTAL_STEPS` → skip（重跑此 cell 不会重训；要继续训需把 `C_TOTAL_STEPS` 改大后重跑此 cell + 上面 config cell）
- `0 < env_step < C_TOTAL_STEPS` → 用 `--resume` 续训（中断重连）
- `env_step == 0` → fresh start

训练时长（L4 / num_envs=6 / 1M steps）≈ 2.5h。Colab Pro+ 5h session 应可吃下两个 phase。

In [ ]:
c_state_path = C_RUN_ROOT / 'trainer_state.json'
if c_state_path.exists():
    c_state = json.loads(c_state_path.read_text(encoding='utf-8'))
    c_current_step = int(c_state.get('env_step', 0))
else:
    c_current_step = 0
print(f'[state] single_cross env_step = {c_current_step:,} / target {C_TOTAL_STEPS:,}')

if c_current_step >= C_TOTAL_STEPS:
    print(f'[skip] single_cross already trained to {c_current_step:,} >= {C_TOTAL_STEPS:,}')
    print('       要继续训:把 C_TOTAL_STEPS 改大,重跑 config cell + 此 cell')
elif c_current_step > 0:
    print(f'[resume] single_cross continuing from {c_current_step:,} -> {C_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {C_RUN_ROOT_STR} \
        --total-steps {C_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {C_MANIFEST_PATH_STR} \
        --device {DEVICE}
else:
    print(f'[train] single_cross fresh start -> {C_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {C_FLOW_PATH} \
        --task-geometry {C_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {C_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {C_MANIFEST_PATH_STR} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {C_RUN_ROOT_STR} \
        --checkpoint-dir {C_CKPT_ROOT_STR}

## 5. Phase 1 — single_cross summary + gate

Gate 5 条与 P1 v6 / tandem / sbs 同口径。Gate summary 落到 `results/single_cross_validation_gate_summary.json`。

In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s1 / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

cross_summary = summarize_phase(
    C_RUN_ROOT,
    C_TOTAL_STEPS,
    'SINGLE_CROSS_VALIDATION',
    'single_cross_validation_gate_summary.json',
)
CROSS_PASS = cross_summary['all_pass']
print()
print(f'CROSS_PASS = {CROSS_PASS}')

## 6. Phase 2 — single_upstream train (1.0M, independent)

**独立运行**：upstream 不门控于 cross。两种 geometry 物理任务不同（侧向跨流 vs 逆流前进），都值得独立看一看。

Skip/resume 逻辑同 single_cross。

In [ ]:
u_state_path = U_RUN_ROOT / 'trainer_state.json'
if u_state_path.exists():
    u_state = json.loads(u_state_path.read_text(encoding='utf-8'))
    u_current_step = int(u_state.get('env_step', 0))
else:
    u_current_step = 0
print(f'[state] single_upstream env_step = {u_current_step:,} / target {U_TOTAL_STEPS:,}')

if u_current_step >= U_TOTAL_STEPS:
    print(f'[skip] single_upstream already trained to {u_current_step:,} >= {U_TOTAL_STEPS:,}')
    print('       要继续训:把 U_TOTAL_STEPS 改大,重跑 config cell + 此 cell')
elif u_current_step > 0:
    print(f'[resume] single_upstream continuing from {u_current_step:,} -> {U_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {U_RUN_ROOT_STR} \
        --total-steps {U_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {U_MANIFEST_PATH_STR} \
        --device {DEVICE}
else:
    print(f'[train] single_upstream fresh start -> {U_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {U_FLOW_PATH} \
        --task-geometry {U_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {U_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {U_MANIFEST_PATH_STR} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {U_RUN_ROOT_STR} \
        --checkpoint-dir {U_CKPT_ROOT_STR}

## 7. Phase 2 — single_upstream summary + gate

In [ ]:
upstream_summary = summarize_phase(
    U_RUN_ROOT,
    U_TOTAL_STEPS,
    'SINGLE_UPSTREAM_VALIDATION',
    'single_upstream_validation_gate_summary.json',
)
UPSTREAM_PASS = upstream_summary['all_pass']
print()
print(f'UPSTREAM_PASS = {UPSTREAM_PASS}')

## 8. Combined verdict + 如何延长训练

总结两个 phase 的 5/5 gate 状态。两个 phase 互不门控；任一 phase fail，单独看其 last100k_mean / peak / OOB 哪条 fail，再决定延长 budget 还是改 reward。

**延长训练的标准流程**（不需要重跑 preflight）：

1. 在 §2 config cell 把对应的 `C_TOTAL_STEPS` 或 `U_TOTAL_STEPS` 改大（如 `1_500_000`）
2. 重跑该 phase 的 config cell（让新值生效）
3. 重跑该 phase 的 train cell — `--resume` 自动续训
4. 重跑该 phase 的 summary cell — gate summary JSON 会被覆盖

**对照预期**：与 tandem/sbs 在严格控制（s1 / k4 / U=1.5 / target=1.5 / arrival_v2 / seed=42 / 1M）下的对比，将填入 `arrival_v2_experiment_report.md` §7.3 修订版。Phase 2 同 benchmark 的 §3 P1 v6 (seed=46/1.5M) 仍保留作为 multi-seed sample 1（P1 v6 + 本 Phase 2 = 同 benchmark 上的 seed=46 / seed=42 二点观测）。

In [ ]:
print('=' * 96)
print('SINGLE U=1.5 CONTROL VALIDATION — combined verdict')
print('-' * 96)
print(f"  CROSS_PASS    = {CROSS_PASS}")
print(f"    final={cross_summary['final_success_rate']:.4f}  "
      f"peak={cross_summary['peak_success_rate']:.4f} @ {cross_summary['peak_step']:,}  "
      f"last100k_mean={cross_summary['last100k_mean_success']:.4f}  "
      f"oob={cross_summary['final_oob_rate']:.4f}")
print(f"  UPSTREAM_PASS = {UPSTREAM_PASS}")
print(f"    final={upstream_summary['final_success_rate']:.4f}  "
      f"peak={upstream_summary['peak_success_rate']:.4f} @ {upstream_summary['peak_step']:,}  "
      f"last100k_mean={upstream_summary['last100k_mean_success']:.4f}  "
      f"oob={upstream_summary['final_oob_rate']:.4f}")
print('=' * 96)

combined = {
    'experiment': 'arrival_v2_single_u15_seed42_1M_control',
    'commit_baseline': 'f179c5b / 1adee4e',
    'seed': SEED,
    'cross': cross_summary,
    'upstream': upstream_summary,
    'cross_pass': CROSS_PASS,
    'upstream_pass': UPSTREAM_PASS,
    'both_pass': bool(CROSS_PASS and UPSTREAM_PASS),
}
out_dir = Path('experiments/arrival_v2_prototype/single_u15_seed42_1M_control_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(combined, indent=2), encoding='utf-8')
print(f'[saved] {out_path}')